Fine-Tune OS LLM for Twin
=========================

Note: intended to be run in [Google Colab](https://colab.research.google.com/) using a A100 runtime.

Training this model on our concatenated dataset can take a few hours. For example, it takes 50 minutes on an A100 GPU.

- **Goal:** fine-tune an open-source model on our custom dataset using LoRA and QLoRA for efficiency.
- **Model:** Llama 3.1 8B, an open-weight model released by Meta.
- **Tools:** fine-tuning performed using the Unsloth library.

Training this model on the concatenated dataset can take a few hours. For example, it takes 50 minutes on an A100 GPU.

# Imports & Settings

In [ ]:
import torch
from trl import SFTTrainer
from datasets import load_dataset, concatenate_datasets
from transformers import TrainingArguments, TextStreamerfrom unsloth
import FastLanguageModel, is_bfloat16_supported

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv(override=True)

MODEL = "meta-llama/Meta-Llama-3.1-8B"

# Steps:

## 1. Log into [Hugging Face](https://huggingface.co/) to access a gated model and upload fine-tuned model to

In [ ]:
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN')

# Log in to HuggingFace
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

## 2. Load Comet ML API key from the .env file

In [ ]:
comet_api_key = os.getenv('COMET_API_KEY')

if comet_api_key and len(comet_api_key)>20:
    print("Comet API key looks good so far")
else:
    print("There might be a problem with your Comet API key?")

## 3. Load the model to fine-tune and its corresponding tokenizer

- Uses Unsloth's `FastLaguageModel` class with the `.from_pretrained()` method.
- Specify model name and max sequence length.
- The `load_in_4bit` argument indicates if we want to use QLoRA (set to `True` for quantized pre-trained weights) or LoRA (set to `False`).

In [ ]:
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=max_seq_length,
    load_in_4bit=False,
)

# 4. Define our LoRA configuration

- Use a rank of 32 that is large enough to imitate the writing style and copy the knowledge from the [instruction samples](../tools/instruction-set-creator.ipynb) or can increase this value to 64 or 128 if results are underwhelming
- Set an alpha of 32, without dropout and without bias, to speed up training.
- Target every linear layer to maximize the quality of the fine-tuning process.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"],
)

## 5. Prepare the data in the right format for fine-tuning

In the [llmtwin](https://huggingface.co/datasets/clanredhead/llmtwin) there are not a lot of samples in the llmtwin dataset (160 samples). This is an issue because the model might not correctly learn the chat template. To address this upsample it with a high-quality general-purpose dataset called [FineTome](https://huggingface.co/datasets/mlabonne/FineTome-Alpaca-100k). This is a filtered version of [arcee-ai/The-Tome](https://huggingface.co/datasets/arcee-ai/The-Tome) using the [fineweb-edu-classifier](https://huggingface.co/HuggingFaceFW/fineweb-edu-classifier). Instead of using the 100,000 samples of this dataset, we will specify we only want 10,000 in the train split. Then concatenate these two datasets to create the final set.

In [ ]:
dataset1 = load_dataset("clanredhead/llmtwin")
dataset2 = load_dataset("mlabonne/FineTome-Alpaca-100k", split="train[:10000]")
dataset = concatenate_datasets([dataset1, dataset2])

# 6. Format data using a chat template

Use the Alpaca template for convenience by mapping all the instructions and answers to the Alpaca template then manually add the end of sentence (EOS) token at the end of each message to ensure that the model learns to output it.

In [ ]:
alpaca_template = """Below is an instruction that describes a task.
Write a response that appropriately completes the request.

### Instruction:
{}
### Response:
{}"""
EOS_TOKEN = tokenizer.eos_token
dataset = dataset.map(format_samples, batched=True, remove_columns=dataset.column_names)

## 7. Divide into training (95%) and test (5%) sets for validation during training

In [ ]:
dataset = dataset.train_test_split(test_size=0.05)

# 8. Fine-tune the model using appropiraite hyperparameters

-  `SFTTrainer()` class stores all the hyperparameters for our training
-  Provide the model, tokenizer, LoRA configuration, and datasets.
- Set a **learning rate of 3e-4** with a **linear scheduler** and a **maximum sequence length of 2048**.
- Train this model for **3 epochs** with a **batch size of 2** and **8 gradient accumulation steps** (for an effective batch size of 16).
- Use the **adamw_8bit optimizer** with a **`weight_decay` of 0.01**.
- Depending on the GPU used, it will automatically use **FP16 or BF16 for the activations**.
- Report training run to Comet ML for experiment tracking.

<img src="../content/LLM-Twin-Fine-Tuning-Comet-ML.jpg" title="Four monitored metrics during fine-tuning in Comet ML" />

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        learning_rate=3e-4,
        lr_scheduler_type="linear",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=3,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        warmup_steps=10,
        output_dir="output",
        report_to="comet_ml",
        seed=0,
    ),
)

trainer.train()

# 9. Test the fine-tuned model

The goal is not to properly evaluate the fine-tuned model, but to make sure that there are no obvious errors related to the tokenizer or chat template.

For fast inference, use `FastLanguageModel.for_inference()` from Unsloth. Directly format an instruction with the Alpaca format.

Note: provide an empty answer to append the assistant header (### Response): at the end of the user instruction. This forces the model to answer the instruction instead of completing it. Also use a text streamer to stream the generation instead of waiting for it to be complete before printing it.

Answer provided by model was Tcorrect and properly formatted with the Alpaca chat template:

> Supervised fine-tuning is a method used to enhance a language model
by providing it with a curated dataset of instructions and their
corresponding answers. This process is designed to align the model's
responses with human expectations, thereby improving its accuracy
and relevance. The goal is to ensure that the model can respond
effectively to a wide range of queries, making it a valuable tool
for applications such as chatbots and virtual assistants.

In [ ]:
FastLanguageModel.for_inference(model)
message = alpaca_prompt.format("Write a paragraph to introduce supervised fine-tuning.", "")
inputs = tokenizer([message], return_tensors="pt").to("cuda")
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer=text_streamer, max_new_tokens=256, use_cache=True)

## 10. Save fine-tuned model locally and push it to the Hugging Face Hub

In [ ]:
model.save_pretrained_merged("model", tokenizer, save_method="merged_16bit")
model.push_to_hub_merged("clanredhead/TwinLlama-3.1-8B", tokenizer,
save_method="merged_16bit")